# KATS — Experiment 3: Attack Survivability Rate

KATS Framework — Kinetic Attack Triage System


In [ ]:
import time

# Three attack scenarios from your research plan
scenarios = {
    'S1-Precision':    dict(az_destroyed=1, bw_loss=0.30, window_min=45,  label='S1: Precision Strike'),
    'S2-Gulf-Strike':  dict(az_destroyed=2, bw_loss=0.60, window_min=20,  label='S2: Coordinated Gulf Strike (Mar 2026)'),
    'S3-Cascading':    dict(az_destroyed=3, bw_loss=0.85, window_min=8,   label='S3: Cascading Collapse'),
}

# Use KATS-SYN as the service registry (15,000 services)
df_scenario = df_kats_syn.copy()
df_scenario['priority_label_enc'] = df_scenario['priority_label'].map(label_map)
X_sc, y_sc = prepare_xy(df_scenario)

# Get KATS-Ensemble ranked predictions (probability of High)
proba_kats  = kats_pipe.predict_proba(X_sc)[:, 2]   # P(High)
proba_dt    = dt_pipe.predict_proba(X_sc)[:, 2]
proba_lr    = lr_pipe.predict_proba(X_sc)[:, 2]

# Rule-based scores
score_b1    = df_scenario['service_criticality'] / 10
score_b3    = (0.5 * df_scenario['service_criticality']/10 +
               0.3 * (1 - df_scenario['rto_minutes'].clip(0,1440)/1440) +
               0.2 * df_scenario['az_risk_score'])

e3_results = []

for sc_key, sc in scenarios.items():
    bw_available   = 1.0 - sc['bw_loss']
    window_min     = sc['window_min']
    total_bw_cap   = df_scenario['bandwidth_required_mbps'].sum() * bw_available

    # Migratable services: those whose data can be transferred within the window
    # data_volume_gb / (bandwidth_required_mbps / 8 / 1000) = transfer_time_min
    df_scenario['transfer_time_min'] = (
        df_scenario['data_volume_gb'] * 8 * 1000 /
        df_scenario['bandwidth_required_mbps'].clip(0.01)
    )
    df_scenario['migratable'] = df_scenario['transfer_time_min'] <= window_min

    n_total_high   = (df_scenario['priority_label'] == 'High').sum()
    n_migratable   = df_scenario['migratable'].sum()

    for method_name, scores in [
        ('KATS-Ensemble',   proba_kats),
        ('B5-DecTree',      proba_dt),
        ('B4-LogReg',       proba_lr),
        ('B1-Criticality',  score_b1.values),
        ('B3-Composite',    score_b3.values),
    ]:
        # Rank all services by score descending
        rank_order = np.argsort(-scores)

        # Greedily select services until bandwidth exhausted or window closed
        bw_used      = 0.0
        rescued_high = 0
        total_selected = 0
        bw_cap_mbps  = total_bw_cap

        for idx in rank_order:
            if not df_scenario['migratable'].iloc[idx]:
                continue
            bw_req = df_scenario['bandwidth_required_mbps'].iloc[idx]
            if bw_used + bw_req > bw_cap_mbps:
                continue
            bw_used += bw_req
            total_selected += 1
            if df_scenario['priority_label'].iloc[idx] == 'High':
                rescued_high += 1

        survivability = rescued_high / max(n_total_high, 1)

        e3_results.append({
            'Scenario':        sc['label'],
            'Method':          method_name,
            'Window_min':      window_min,
            'BW_Loss_pct':     int(sc['bw_loss']*100),
            'Total_High':      n_total_high,
            'Rescued_High':    rescued_high,
            'Survivability':   round(survivability, 4),
            'Services_Migrated': total_selected,
        })

df_e3 = pd.DataFrame(e3_results)
df_e3.to_csv('/kaggle/working/experiment3_results.csv', index=False)

print("=" * 85)
print("EXPERIMENT 3 — ATTACK SCENARIO ANALYSIS (Survivability Rate)")
print("=" * 85)
pivot_e3 = df_e3.pivot(index='Method', columns='Scenario', values='Survivability')
pivot_e3['Mean'] = pivot_e3.mean(axis=1)
pivot_e3 = pivot_e3.sort_values('Mean', ascending=False)
print(pivot_e3.round(4).to_string())

print("\n📊 KEY RESULT — S2 Gulf Strike (your headline):")
s2 = df_e3[df_e3['Scenario'].str.contains('S2')].sort_values('Survivability', ascending=False)
print(s2[['Method','Survivability','Rescued_High','Total_High','Services_Migrated']].to_string(index=False))

In [ ]:
import time
import warnings
warnings.filterwarnings('ignore')

scenarios = {
    'S1-Precision':   dict(az_destroyed=1, bw_loss=0.30, window_min=45,  label='S1: Precision Strike'),
    'S2-Gulf-Strike': dict(az_destroyed=2, bw_loss=0.60, window_min=20,  label='S2: Coordinated Gulf Strike (Mar 2026)'),
    'S3-Cascading':   dict(az_destroyed=3, bw_loss=0.85, window_min=8,   label='S3: Cascading Collapse'),
}

df_sc = df_kats_syn.copy()
X_sc, y_sc = prepare_xy(df_sc)

# Pre-compute model scores
proba_kats = kats_pipe.predict_proba(X_sc)[:, 2]
proba_dt   = dt_pipe.predict_proba(X_sc)[:, 2]
proba_lr   = lr_pipe.predict_proba(X_sc)[:, 2]
score_b1   = df_sc['service_criticality'].values / 10.0
score_b3   = (0.5 * df_sc['service_criticality']/10 +
              0.3 * (1 - df_sc['rto_minutes'].clip(0,1440)/1440) +
              0.2 * df_sc['az_risk_score']).values

# ── FIXED transfer time ──────────────────────────────────────────────────
# Formula: size_Mb / bandwidth_Mbps = seconds → divide by 60 for minutes
df_sc['size_mb']      = df_sc['data_volume_gb'] * 1024          # GB → MB (= Mb at 8-bit)
df_sc['bw_mbps']      = df_sc['bandwidth_required_mbps'].clip(0.5, None)   # min 0.5 Mbps

e3_results = []

for sc_key, sc in scenarios.items():
    bw_factor    = 1.0 - sc['bw_loss']
    window_min   = sc['window_min']

    # Effective per-service bandwidth under attack
    df_sc['eff_bw']            = df_sc['bw_mbps'] * bw_factor
    df_sc['transfer_time_min'] = (df_sc['size_mb'] / df_sc['eff_bw']) / 60.0
    df_sc['migratable']        = (
        (df_sc['transfer_time_min'] <= window_min) |
        (df_sc['data_volume_gb'] <= 2.0)           # tiny services always fit
    )

    n_migratable = df_sc['migratable'].sum()
    n_total_high = (df_sc['priority_label'] == 'High').sum()

    for method_name, scores in [
        ('KATS-Ensemble',  proba_kats),
        ('B5-DecTree',     proba_dt),
        ('B4-LogReg',      proba_lr),
        ('B1-Criticality', score_b1),
        ('B3-Composite',   score_b3),
    ]:
        rank_order   = np.argsort(-scores)
        bw_budget    = df_sc['bw_mbps'].sum() * bw_factor  # total available Mbps
        bw_used      = 0.0
        rescued_high = 0
        total_sel    = 0

        for idx in rank_order:
            if not df_sc['migratable'].iloc[idx]:
                continue
            bw_req = df_sc['eff_bw'].iloc[idx]
            if bw_used + bw_req > bw_budget:
                continue
            bw_used += bw_req
            total_sel += 1
            if df_sc['priority_label'].iloc[idx] == 'High':
                rescued_high += 1

        survivability = rescued_high / max(n_total_high, 1)
        e3_results.append({
            'Scenario':          sc['label'],
            'Method':            method_name,
            'Window_min':        window_min,
            'BW_Loss_pct':       int(sc['bw_loss']*100),
            'AZs_Destroyed':     sc['az_destroyed'],
            'Total_High':        n_total_high,
            'Rescued_High':      rescued_high,
            'Survivability':     round(survivability, 4),
            'Services_Migrated': total_sel,
            'Migratable_Pool':   n_migratable,
        })

df_e3_fixed = pd.DataFrame(e3_results)
df_e3_fixed.to_csv('/kaggle/working/experiment3_results_fixed.csv', index=False)

print("=" * 80)
print("EXPERIMENT 3 FIXED — SURVIVABILITY RATE (Higher = Better)")
print("=" * 80)
pivot_e3 = df_e3_fixed.pivot(index='Method', columns='Scenario', values='Survivability')
pivot_e3.columns.name = None
pivot_e3['Mean'] = pivot_e3.mean(axis=1)
print(pivot_e3.sort_values('Mean', ascending=False).round(4).to_string())

print("\n📊 S2 Gulf Strike — Your Headline Result:")
s2 = df_e3_fixed[df_e3_fixed['Scenario'].str.contains('S2')].sort_values('Survivability', ascending=False)
print(s2[['Method','Survivability','Rescued_High','Total_High','Services_Migrated']].to_string(index=False))